In [3]:
# !pip --version

In [4]:
# !pip install dotenv

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import os 
print(f"키값은:{os.environ['OPENAI_API_KEY'][:8]}")

키값은:sk-proj-


In [7]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH02-Prompt")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [8]:
from langchain_openai import ChatOpenAI

llm =  ChatOpenAI()

In [9]:
from langchain_core.prompts import PromptTemplate

#Template 정의. {country}는 변수로, 이후에 값이 들어갈 자리를 의미
template = "{country}의 수도는 어디인가요?"

#from template 메서드를 이용하여 PromptTemplate 객체 생성
prompt =  PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [10]:
#format 메서드 사용하여 변수에 값 넣기
prompt = prompt.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [11]:
#from_template 메서드를 이용하여 PromptTemplate 객체 생성
template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate.from_template(template)

chain = prompt | llm

chain.invoke("대한민국").content

'대한민국의 수도는 서울입니다.'

In [12]:
#PromptTemplate 객체를 활용하여 prompt_template 생성
template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate(
    template=template,
    input_variables=["country"]
)

prompt

prompt.format(country="대한민국")

'대한민국의 수도는 어디인가요?'

In [13]:
#partial variables
template = "{country1}과 {country2}의 수도는 각각 어디인가요?"

prompt = PromptTemplate(
    template=template,
    input_variables=["country1"],
    partial_variables={
        "country2": "미국"
    }
)

prompt


PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '미국'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [14]:

prompt.format(country1="대한민국")

'대한민국과 미국의 수도는 각각 어디인가요?'

In [15]:
#prompt_partial로 변수 미리 채워두기
prompt_partial = prompt.partial(country2="캐나다")
prompt_partial

prompt_partial.format(country1="대한민국")

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [16]:
chain = prompt_partial | llm
chain.invoke("대한민국").content

'대한민국의 수도는 서울이며, 캐나다의 수도는 오타와입니다.'

In [17]:
#partial()로 설정된 값도 새로운 값으로 설정될 수 있음
chain.invoke({"country1": "대한민국", "country2": "호주"}).content

'대한민국의 수도는 서울이고 호주의 수도는 캔버라입니다.'

In [18]:
##부분변수 활용하기

from datetime import datetime

datetime.now().strftime("%B %d")

'September 15'

In [19]:
def get_today():
    return datetime.now().strftime("%B %d")

In [20]:
prompt = PromptTemplate(
    template="오늘의 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해 주세요.",
    input_variables=["n"],
    partial_variables={
        "today": get_today
    },
)

In [21]:
prompt.format(n=3)

'오늘의 날짜는 September 15입니다. 오늘이 생일인 유명인 3명을 나열해 주세요. 생년월일을 표기해 주세요.'

In [22]:
chain = prompt |llm
print(chain.invoke(3).content)

1. Prince Harry (1984년 9월 15일)
2. Tommy Lee Jones (1946년 9월 15일)
3. Chelsea Kane (1988년 9월 15일)


In [23]:
## YAML 파일로부터 프롬프트 템플릿 로드하기 

from langchain_core.prompts import load_prompt
prompt = load_prompt("fruit_color.yaml", encoding="utf-8")
prompt

C:\Users\user\AppData\Local\Temp\ipykernel_10868\3461920716.py:4: LangChainDeprecationWarning: The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt = load_prompt("fruit_color.yaml", encoding="utf-8")


PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색깔이 뭐야?')

In [24]:
prompt.format(fruit="사과")

'사과의 색깔이 뭐야?'

In [25]:

prompt2 = load_prompt("capital.yaml", encoding="utf-8")
print(prompt2.format(country="대한민국"))

대한민국의 수도에 대해서 알려주세요.
수도의 특징을 다음의 양식에 맞게 정리해 주세요.
300자 내외로 작성해 주세요.
한글로 작성해 주세요.
----
[양식]
1. 면적
2. 인구
3. 역사적 장소
4. 특산품

#Answer:



In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote.messages import stream_response

chain = prompt2 | ChatOpenAI(model_name="gpt-4o", temperature=0) | StrOutputParser()

answer = chain.stream({"country": "대한민국"})
stream_response(answer)

1. 면적: 대한민국의 수도인 서울특별시는 약 605.21㎢의 면적을 가지고 있습니다. 이는 대한민국 전체 면적의 약 0.6%에 해당합니다.
2. 인구: 서울의 인구는 약 950만 명으로, 대한민국에서 가장 인구가 많은 도시입니다. 다양한 문화와 경제 활동의 중심지로서 많은 사람들이 거주하고 있습니다.
3. 역사적 장소: 서울에는 경복궁, 창덕궁, 덕수궁 등 조선시대의 궁궐들이 있으며, 이외에도 한양도성, 종묘 등 유네스코 세계문화유산으로 지정된 역사적 장소들이 많습니다.
4. 특산품: 서울은 전통과 현대가 조화를 이루는 도시로, 한복, 한지 공예품, 전통 음식인 김치와 떡 등이 유명합니다. 또한, 다양한 현대 패션과 기술 제품들도 서울의 특산품으로 꼽힙니다.

In [27]:
##ChatPromptTemplate은 대화형 챗봇과 같이 프롬프트 템플릿보다 더 자연스럽고 좋은 답변을 얻을 수 있음 
##system, human, ai 셋 중 하나의 역할은 반드시 들어가야함 
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template("{country}의 수도는 어디인가요?")
chat_prompt

ChatPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?'), additional_kwargs={})])

In [28]:
#human으로 자동설정되는 케이스
chat_prompt.format(country = "대한민국")

'Human: 대한민국의 수도는 어디인가요?'

In [29]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name}입니다."),
        ("human", "반가워요!"),
        ("ai", "안녕하세요! 무엇을 도와드릴까요?"),
        ("human", "{user_input}"),
    ]
)

In [30]:
messages = chat_template.format_messages(
    name="가윤", user_input="당신의 이름은 무엇입니까?"
)
messages

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 가윤입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='반가워요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신의 이름은 무엇입니까?', additional_kwargs={}, response_metadata={})]

In [31]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm.invoke(messages).content

'제 이름은 가윤입니다. 저에게 궁금한 점이 있거나 도움이 필요하시면 언제든지 말씀해주세요!'

In [32]:
chain = chat_template | llm

chain.invoke({"name": "Gayoon", "user_input": "당신의 이름은 무엇입니까?"}).content

'제 이름은 Gayoon입니다. 어떻게 도와드릴까요?'

In [33]:
#MessagesPlaceHolder 아직 채워지지 않았지만 나중에 채워질 메세지를 위한 임시 자리
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.",
        ),
        MessagesPlaceholder(variable_name="conversation"),
        ("human", "지금까지의 대화를 {word_count} 단어로 요약합니다."),
    ]
)
chat_prompt

ChatPromptTemplate(input_variables=['conversation', 'word_count'], input_types={'conversation': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annota

In [34]:
formatted_chat_prompt = chat_prompt.format(
    word_count=5,
    conversation=[
        ("human", "안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다."),
        ("ai", "반가워요! 앞으로 잘 부탁드립니다."),
    ]
)
print(formatted_chat_prompt)

System: 당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.
Human: 안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.
AI: 반가워요! 앞으로 잘 부탁드립니다.
Human: 지금까지의 대화를 5 단어로 요약합니다.


In [35]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

chain = chat_prompt | llm | StrOutputParser()

In [36]:
chain.invoke(
    {
        "word_count": 5,
        "conversation": [
            (
                "human",
                "안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.",

            ),
            ("ai", "반가워요! 앞으로 잘 부탁드립니다."),
        ]
    }
)


'새로 입사한 테디, 만나서 반가워요!'

In [37]:
import os
from dotenv import load_dotenv

load_dotenv()

print("OPENAI:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")

OPENAI: sk-proj-...
LANGSMITH: lsv2_pt_...
